# Telescope Mount Keyboard Map

Interactive Plotly version — tweak `key_map`, `CATEGORY_COLORS`, or the row offsets in the cells below and re-run to see the change.

Requires: `pip install plotly`


In [38]:
# --- Config: keymap + category colors ---
import re
import plotly.graph_objects as go
import nbformat

key_map = {
    'ArrowLeft': 'moveLeft', 'ArrowRight': 'moveRight', 'ArrowUp': 'moveUp', 'ArrowDown': 'moveDown',
    ',': 'moveCCW', '.': 'moveCW',
    'a': 'moveLeft', 'd': 'moveRight', 'w': 'moveUp', 's': 'moveDown', 'q': 'moveCCW', 'e': 'moveCW',
    '1': 'speed1', '2': 'speed2', '3': 'speed3', '4': 'speed4', '5': 'speed5',
    '6': 'speed6', '7': 'speed7', '8': 'speed8', '9': 'speed9',
    'f': 'referenceFrame', 'g': 'centerGrid', 'h': 'findHome', 'k': 'abortSlew',
    'r': 'resetSP', 't': 'toggleTracking', 'p': 'togglePark',
    'Escape': 'abortSlew', 'Backspace': 'abortSlew',
    'z': 'dashboard', 'x': 'alignment', 'c': 'connect', 'v': 'settings',
    'l': 'log', 'n': 'nearby',
}

# Tweak these to change the palette
CATEGORY_COLORS = {
    'movement': '#4C8BF5',
    'speed':    '#34A853',
    'action':   '#FB8C00',
    'abort':    '#E53935',
    'nav':      '#8E5CD9',
    'plain':    '#DADCE0',
}

MOVEMENT_CMDS = {'moveLeft', 'moveRight', 'moveUp', 'moveDown', 'moveCCW', 'moveCW'}
ACTION_CMDS   = {'referenceFrame', 'centerGrid', 'findHome', 'resetSP', 'toggleTracking', 'togglePark'}
NAV_CMDS      = {'dashboard', 'alignment', 'connect', 'settings', 'nearby', 'log'}

def category_for(cmd):
    if cmd is None:
        return 'plain'
    if cmd in MOVEMENT_CMDS:
        return 'movement'
    if cmd.startswith('speed'):
        return 'speed'
    if cmd == 'abortSlew':
        return 'abort'
    if cmd in ACTION_CMDS:
        return 'action'
    if cmd in NAV_CMDS:
        return 'nav'
    return 'plain'

DISPLAY_CHAR = {
    'ArrowLeft': '\u2190', 'ArrowRight': '\u2192', 'ArrowUp': '\u2191', 'ArrowDown': '\u2193',
    'Escape': 'Esc', 'Backspace': 'Bksp',
}

def label_for(raw_key):
    return DISPLAY_CHAR.get(raw_key, raw_key.upper() if len(raw_key) == 1 else raw_key)

def split_camel(cmd):
    """Split long camelCase commands onto two lines so they fit on a keycap."""
    m = re.match(r'^([a-z]+)([A-Z].*)$', cmd)
    if m and len(cmd) > 2:
        return m.group(1), m.group(2)
    return cmd, None


In [39]:
# --- Geometry ---
# Row offsets are independent of the leading modifier key's width (Tab/Caps/Shift).
# That's what caused misalignment before: the row offset must be a small fixed
# stagger from the digit row, and the modifier key is drawn *ending* at that
# offset (so it overhangs left), not the row starting *after* the modifier.

GAP = 0.08
KH = 1.0

QWERTY_OFFSET = 0.5   # tweak these three to change the stagger amount
ASDF_OFFSET = 0.75
ZXCV_OFFSET = 1.25

row_defs = [
    # y, start_x, [(raw_key_or_None, width), ...]
    (4.15, -1.0, [
        ('Escape', 1.0),   # Escape shares the digit row, small gap after it
        ('1', 1), ('2', 1), ('3', 1), ('4', 1), ('5', 1), ('6', 1), ('7', 1), ('8', 1), ('9', 1), ('0', 1), ('-', 1), ('=', 1),
        ('Backspace', 2),
    ]),
    (3.0, QWERTY_OFFSET - 1.5, [
        ('Tab', 1.5), ('q', 1), ('w', 1), ('e', 1), ('r', 1), ('t', 1), ('y', 1), ('u', 1), ('i', 1), ('o', 1), ('p', 1), ('[', 1), (']', 1), ('\\', 1.5)
    ]),
    (1.85, ASDF_OFFSET - 1.75, [
        ('Caps', 1.75), ('a', 1), ('s', 1), ('d', 1), ('f', 1), ('g', 1), ('h', 1), ('j', 1), ('k', 1), ('l', 1), (';', 1), ("'", 1),
        ('Enter', 2.25),
    ]),
    (0.7, ZXCV_OFFSET - 2.25, [
        ('Shift', 2.25), ('z', 1), ('x', 1), ('c', 1), ('v', 1), ('b', 1), ('n', 1), ('m', 1), (',', 1), ('.', 1), ('/', 1),
        ('Shift', 2.0),
    ]),
    (-0.45, 0.0, [
        ('Ctrl', 1.5), ('Win', 1.25), ('Alt', 1.25), ('Space', 4.0), ('Alt', 1.25), ('Win', 1.25), ('Menu', 1.25), ('Ctrl', 1.5),
    ]),
]

ARROW_X0 = 14  # x position of the arrow-key cluster, right of the main block

def add_key(shapes, annotations, x, y, w, h, raw_key):
    if raw_key is None:
        return
    cmd = key_map.get(raw_key)
    cat = category_for(cmd)
    face = CATEGORY_COLORS[cat]
    text_color = 'white' if cat != 'plain' else '#5F6368'

    x0, x1 = x, x + w - GAP
    y0, y1 = y, y + h - GAP
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2

    shapes.append(dict(
        type='rect', x0=x0, y0=y0, x1=x1, y1=y1,
        line=dict(color='#3c4043', width=1.2),
        fillcolor=face, layer='below',
    ))

    if cmd:
        line1, line2 = split_camel(cmd)
        char_y = cy + (0.16 if line2 else 0.22)
        annotations.append(dict(x=cx, y=char_y, text=f'<b>{label_for(raw_key)}</b>',
                                 showarrow=False, font=dict(size=15, color=text_color), align='center'))
        if line2:
            annotations.append(dict(x=cx, y=cy - 0.15, text=f'<i>{line1}</i>',
                                     showarrow=False, font=dict(size=9, color=text_color), align='center'))
            annotations.append(dict(x=cx, y=cy - 0.33, text=f'<i>{line2}</i>',
                                     showarrow=False, font=dict(size=9, color=text_color), align='center'))
        else:
            annotations.append(dict(x=cx, y=cy - 0.24, text=f'<i>{cmd}</i>',
                                     showarrow=False, font=dict(size=9.5, color=text_color), align='center'))
    else:
        annotations.append(dict(x=cx, y=cy, text=label_for(raw_key),
                                 showarrow=False, font=dict(size=12, color=text_color), align='center'))


In [40]:
# --- Build and render the figure ---
shapes, annotations = [], []

for y, start_x, keys in row_defs:
    x = start_x
    for raw_key, w in keys:
        add_key(shapes, annotations, x, y, w, KH, raw_key)
        x += w

# Arrow cluster
add_key(shapes, annotations, ARROW_X0 + 1.0, 0.7, 1.0, KH, 'ArrowUp')
add_key(shapes, annotations, ARROW_X0 + 0.0, -0.45, 1.0, KH, 'ArrowLeft')
add_key(shapes, annotations, ARROW_X0 + 1.0, -0.45, 1.0, KH, 'ArrowDown')
add_key(shapes, annotations, ARROW_X0 + 2.0, -0.45, 1.0, KH, 'ArrowRight')

# Legend
legend_items = [
    ('movement', 'Movement (slew)'),
    ('speed', 'Speed select'),
    ('action', 'Mount actions'),
    ('abort', 'Abort / stop'),
    ('nav', 'Navigate pages'),
]
ly = 5.7
for i, (cat, text) in enumerate(legend_items):
    swatch_x = i * 3.6
    shapes.append(dict(type='rect', x0=swatch_x, y0=ly, x1=swatch_x + 0.5, y1=ly + 0.4,
                        line=dict(color='#3c4043', width=1), fillcolor=CATEGORY_COLORS[cat], layer='below'))
    annotations.append(dict(x=swatch_x + 0.65, y=ly + 0.2, text=text, showarrow=False,
                             font=dict(size=11, color='#202124'), align='left', xanchor='left'))

annotations.append(dict(x=0, y=7.15, text='<b>Alpaca Pilot Keyboard Map</b>', showarrow=False,
                         font=dict(size=22, color='#202124'), align='left', xanchor='left'))
annotations.append(dict(x=0, y=6.75,
                         text='WASD / arrows share movement bindings; comma/period and Q/E control roll axis rotation.',
                         showarrow=False, font=dict(size=11, color='#5F6368'), align='left', xanchor='left'))

fig = go.Figure()
fig.update_layout(
    shapes=shapes,
    annotations=annotations,
    xaxis=dict(visible=False, range=[-1.5, 18.0]),
    yaxis=dict(visible=False, range=[-1.2, 7.7], scaleanchor='x', scaleratio=1),
    width=1500, height=680,
    plot_bgcolor='white', paper_bgcolor='white',
    margin=dict(l=10, r=10, t=10, b=10),
)

fig.show()

# To export a static PNG instead (requires Chrome, installed once via `plotly_get_chrome`):
# fig.write_image('keyboard-map.png', scale=2)
